# Day 6 — Linear Regression From Scratch

## 1. Learning Objectives
- Put together everything from Days 1-5.
- Implement Linear Regression without PyTorch's `nn` module.
- Write a manual training loop.
- Compare a Numpy implementation vs a PyTorch Autograd implementation.

## 2. Prerequisites
- Tensor operations and Shapes (Days 3 & 4).
- Autograd and backward passes (Day 5).

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

## 3. Concept Explanation
Linear Regression finds the line of best fit for a set of data points. The formula is $y = w \cdot x + b$.
- $w$ is the weight (slope).
- $b$ is the bias (intercept).

We start with random $w$ and $b$. We calculate our predictions, find the error (Loss), calculate how the error changes with respect to $w$ and $b$ (Gradients), and update them slightly to reduce the error. This is a **Training Loop**.

## 4. Why This Matters
Every Neural Network, no matter how massive, follows this exact same loop: Forward Pass $\rightarrow$ Loss $\rightarrow$ Backward Pass $\rightarrow$ Update. If you can build Linear Regression from scratch, you conceptually understand Deep Learning.

In [ ]:
# Setup: Create dummy data
# Let's say true weight is 2.5 and true bias is 1.0
x_data = torch.rand(100, 1) * 10
y_true = 2.5 * x_data + 1.0 + torch.randn(100, 1) * 2.0 # Add some noise

plt.scatter(x_data, y_true)
plt.title("Dummy Data")
plt.show()

## 8. Simple Example: The PyTorch Way
Let's initialize our random weight and bias, and set `requires_grad=True`.

In [ ]:
w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

learning_rate = 0.01

# 9. Code Walkthrough: The Manual Training Loop
for epoch in range(100):
    # 1. Forward Pass
    y_pred = x_data * w + b
    
    # 2. Calculate Loss (Mean Squared Error)
    loss = ((y_pred - y_true) ** 2).mean()
    
    # 3. Backward Pass
    loss.backward()
    
    # 4. Update Weights (Wrap in torch.no_grad() because we don't want to track this update)
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        
        # 5. Zero out gradients for the next loop!
        w.grad.zero_()
        b.grad.zero_()
        
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}, w = {w.item():.4f}, b = {b.item():.4f}")

print(f"\nFinal Learned w: {w.item():.4f} (True was 2.5)")
print(f"Final Learned b: {b.item():.4f} (True was 1.0)")

## 10. Experiment: What happens without Autograd?
If we used NumPy, we would have to calculate the derivatives manually.
$\frac{d(Loss)}{dw} = \frac{2}{N} \sum (y_{pred} - y_{true}) \cdot x$

PyTorch calculated this implicitly during `loss.backward()`. This is why PyTorch is a game-changer.

## 11. Practice Exercise 1: Modifying the Learning Rate
Copy the training loop above. Change the learning rate to `0.5`. What happens to the loss? Why?

In [ ]:
# Write your code here

In [ ]:
# SOLUTION
# If you set lr=0.5, the loss will explode to infinity (NaN).
# This is called "exploding gradients" or "overshooting the minimum". 
# The step taken was too large.

## 13. Debugging Challenge
A junior developer wrote this training loop, but the model isn't learning. The loss goes down once, and then behaves weirdly. Find the bug.

In [ ]:
w_buggy = torch.randn(1, requires_grad=True)
b_buggy = torch.randn(1, requires_grad=True)

for epoch in range(10):
    y_pred = x_data * w_buggy + b_buggy
    loss = ((y_pred - y_true) ** 2).mean()
    loss.backward()
    
    with torch.no_grad():
        w_buggy -= 0.01 * w_buggy.grad
        b_buggy -= 0.01 * b_buggy.grad
        
    print(loss.item())

**Solution:** They forgot to zero out the gradients! `w_buggy.grad.zero_()` and `b_buggy.grad.zero_()` must be called at the end of the `with torch.no_grad():` block, otherwise the gradients from epoch 1 add to epoch 2, and so on.

## 17. Interview Questions
1. **Why do we need `torch.no_grad()` when updating weights manually?**
   *Answer*: Weight updates are a mathematical operation. If we don't wrap them in `no_grad()`, PyTorch will track the update operation in the computational graph, which we don't want. We only want to update the data values.
2. **What are the 5 core steps of a standard training loop?**
   *Answer*: 1) Forward pass, 2) Loss calculation, 3) Backward pass (`loss.backward()`), 4) Weight update, 5) Zero gradients.

## 19. Day Summary
- You built your first ML model completely from scratch!
- The training loop sequence is sacred: Forward $\rightarrow$ Loss $\rightarrow$ Backward $\rightarrow$ Step $\rightarrow$ Zero.
- The Learning Rate is highly sensitive; too small = slow learning, too big = exploding loss.